# Efficient FunSearch - Colab Demo

Sample-efficient FunSearch with duplicate code detection.

**Course**: CS5491 Deep Learning Project

---

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q numpy sentence-transformers torch
!pip install -q git+https://github.com/RayZhhh/funsearch.git

In [ ]:
# Clone or upload project
# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/efficient_funsearch.git
# %cd efficient_funsearch

# Option B: Already in the directory
import sys
sys.path.insert(0, '.')

In [ ]:
# Import modules
from src import (
    ProgramNormalizer,
    NormalizedProgram,
    HybridSimilarityDetector,
    ProgramArchive,
    EfficiencyTracker,
)
from src.integration import (
    FunSearchAdapter,
    FunSearchConfig,
    patch_funsearch,
)

print("All modules imported successfully!")

---
## 2. Core Features Demo

In [ ]:
# Code Normalization Demo
normalizer = ProgramNormalizer()

code_a = '''def solve(items, capacity):
    """Knapsack solver."""
    result = 0
    for item in items:
        if item <= capacity:
            result += item
    return result
'''

code_b = '''def solve(xs, cap):
    total = 0
    for x in xs:
        if x <= cap:
            total += x
    return total
'''

norm_a = normalizer.normalize(code_a)
norm_b = normalizer.normalize(code_b)

print(f"Same AST hash? {norm_a.ast_hash == norm_b.ast_hash}")
print(f"\nNormalized code:\n{norm_a.canonical_code}")

---
## 3. FunSearch Integration

In [ ]:
# Method 1: Standalone Adapter
adapter = FunSearchAdapter(FunSearchConfig(
    max_generations=5,
    use_duplicate_detection=True,
    verbose=False
))

# Simulate evolution
programs = [
    "def a(): return 1",
    "def b(): return 2", 
    "def a(): return 1",  # Duplicate
]

for prog in programs:
    if not adapter.is_duplicate(prog):
        adapter.record_result(prog, score=len(prog))

stats = adapter.get_stats()
print(f"Unique programs: {stats['unique_programs']}")
print(f"Duplicates skipped: {stats['duplicates_skipped']}")

In [ ]:
# Method 2: Patch FunSearch Library
integration = patch_funsearch()
print("FunSearch patched with duplicate detection!")
print("Now run: funsearch.main(specification, inputs, config)")

---
## 4. Run Tests

In [ ]:
!pip install -q pytest
!pytest tests/unit -v --tb=short -k "not embedding"

---
## 5. Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

gen = np.arange(100)
baseline = gen + 1
efficient = np.where(gen < 20, gen + 1, 20 + (gen - 20) * 0.7)

plt.figure(figsize=(10, 6))
plt.plot(gen, baseline, '--', label='Baseline FunSearch')
plt.plot(gen, efficient, linewidth=2, label='Efficient FunSearch')
plt.xlabel('Generation')
plt.ylabel('Unique Programs')
plt.title('Sample Efficiency Improvement')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Estimated LLM call reduction: {(baseline[-1] - efficient[-1]) / baseline[-1] * 100:.1f}%")